Let's really quickly recall what an activation is, and what tensor parallelism is trying to solve.

In [15]:
import torch

batch, seq_len, in_features, out_features = 2, 3, 4, 6
linear = torch.nn.Linear(in_features=in_features, out_features=out_features)

X = torch.randn(batch, seq_len, in_features)
Y = linear(X)

print(f"Input X shape:  {X.shape}")   # (batch, seq_len, in_features)
print(f"Output Y shape: {Y.shape}")   # (batch, seq_len, out_features)

Input X shape:  torch.Size([2, 3, 4])
Output Y shape: torch.Size([2, 3, 6])


The activation tensor Y has shape (2, 3, 6) = 36 numbers.

During training, Y must be stored in memory for backward pass. This is the activation memory we're trying to reduce.

There are two ways which we can split Y across gpus, column wise parallism and row wise parallism.

This is column wise parallism.

In [9]:
import torch
import torch.nn.functional as F

tp_size = 2           # Number of GPUs in tensor parallel group
tp_rank_1 = 0          # GPU 0
tp_rank_2 = 1          # GPU 1

# Original full weight matrix: (out_features=6, in_features=4)
linear_full = torch.nn.Linear(4, 6, bias=False)

# Split the weight into 2 chunks along output dimension (dim=0)
# Each chunk has shape (3, 4) - half the output features
W_sharded_gpu_1 = linear_full.weight.chunk(tp_size, dim=0)[tp_rank_1]  # GPU 0 gets rows 0-2
W_sharded_gpu_2 = linear_full.weight.chunk(tp_size, dim=0)[tp_rank_2]  # GPU 1 gets rows 3-5


# Input same on both GPUs: (batch=2, seq=3, features=4)
X = torch.randn(2, 3, 4)

# Forward pass - each GPU computes its slice of the output
Y_shard_1 = F.linear(X, W_sharded_gpu_1)  # GPU 0: Y0 = X @ W0.T → shape (2, 3, 3)
Y_shard_2 = F.linear(X, W_sharded_gpu_2)  # GPU 1: Y1 = X @ W1.T → shape (2, 3, 3)

# Reconstruct full output by concatenating along feature dimension
Y_full = torch.cat([Y_shard_1, Y_shard_2], dim=-1)  # shape (2, 3, 6)


print(f"Full weight: {linear_full.weight.shape}")      # (6, 4)
print(f"Shard per GPU: {W_sharded_gpu_1.shape}")       # (3, 4)
print(f"Input X: {X.shape}")                          # (2, 3, 4)
print(f"Y_shard_1 (GPU 0): {Y_shard_1.shape}")        # (2, 3, 3)
print(f"Y_shard_2 (GPU 1): {Y_shard_2.shape}")        # (2, 3, 3)
print(f"Y_full (reconstructed): {Y_full.shape}")   

Full weight: torch.Size([6, 4])
Shard per GPU: torch.Size([3, 4])
Input X: torch.Size([2, 3, 4])
Y_shard_1 (GPU 0): torch.Size([2, 3, 3])
Y_shard_2 (GPU 1): torch.Size([2, 3, 3])
Y_full (reconstructed): torch.Size([2, 3, 6])


Now row parallel

In [10]:
import torch
import torch.nn.functional as F

tp_size = 2           # Number of GPUs in tensor parallel group
tp_rank_1 = 0          # GPU 0
tp_rank_2 = 1          # GPU 1

# Original full weight matrix: (out_features=6, in_features=4)
linear_full = torch.nn.Linear(4, 6, bias=False)

# Split the weight into 2 chunks along INPUT dimension (dim=1)
# Each chunk has shape (6, 2) - half the input features
W_sharded_gpu_1 = linear_full.weight.chunk(tp_size, dim=1)[tp_rank_1]  # GPU 0 gets cols 0-1
W_sharded_gpu_2 = linear_full.weight.chunk(tp_size, dim=1)[tp_rank_2]  # GPU 1 gets cols 2-3

# Input is SPLIT across GPUs along feature dimension (same as weight input dim)
X = torch.randn(2, 3, 4)
X_gpu_1 = X.chunk(tp_size, dim=-1)[tp_rank_1]  # GPU 0 gets features 0-1 → shape (2, 3, 2)
X_gpu_2 = X.chunk(tp_size, dim=-1)[tp_rank_2]  # GPU 1 gets features 2-3 → shape (2, 3, 2)

# Forward pass - each GPU computes partial result
Y_partial_1 = F.linear(X_gpu_1, W_sharded_gpu_1)  # GPU 0: shape (2, 3, 6)
Y_partial_2 = F.linear(X_gpu_2, W_sharded_gpu_2)  # GPU 1: shape (2, 3, 6)

# All-reduce: sum partials to get full output
Y_full = Y_partial_1 + Y_partial_2  # shape (2, 3, 6)

print(f"Full weight: {linear_full.weight.shape}")      # (6, 4)
print(f"Shard per GPU: {W_sharded_gpu_1.shape}")       # (6, 2)
print(f"X_gpu_1: {X_gpu_1.shape}")                     # (2, 3, 2)
print(f"X_gpu_2: {X_gpu_2.shape}")                     # (2, 3, 2)
print(f"Y_partial_1 (GPU 0): {Y_partial_1.shape}")     # (2, 3, 6)
print(f"Y_partial_2 (GPU 2): {Y_partial_2.shape}")     # (2, 3, 6)
print(f"Y_full (after sum): {Y_full.shape}")           # (2, 3, 6)

Full weight: torch.Size([6, 4])
Shard per GPU: torch.Size([6, 2])
X_gpu_1: torch.Size([2, 3, 2])
X_gpu_2: torch.Size([2, 3, 2])
Y_partial_1 (GPU 0): torch.Size([2, 3, 6])
Y_partial_2 (GPU 2): torch.Size([2, 3, 6])
Y_full (after sum): torch.Size([2, 3, 6])


## Parallel In Classes

Now that you've seen the tensor parallel in it's most simplest form, we should probably turn them into classes.

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class ColumnParallelLinear(nn.Module):

    def __init__(self, in_features, out_features, tp_size, tp_rank, gather_output=False):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.tp_size = tp_size
        self.tp_rank = tp_rank
        self.gather_output = gather_output

        # Each rank owns only part of the output rows.
        assert out_features % tp_size == 0
        self.out_per_rank = out_features // tp_size

        # Local shard: (out_per_rank, in_features)
        self.weight = nn.Parameter(torch.empty(self.out_per_rank, in_features))

    
    def forward(self, x):
        y_local = F.linear(x, self.weight)
        return y_local

Right now forward() always returns only the local shard.

That is fine for sharded execution, but sometimes the next layer wants the full output, like layerNorm, so we also need an optional (should we gather after this linear layer) thing

In [17]:
def gather_output_shards(self, y_local):
    
    # Make one empty slot per rank, each same shape as local shard
    shards = [torch.empty_like(y_local) for _ in range(self.tp_size)]

    # Collect y_local from every rank into shards
    dist.all_gather(shards, y_local)

    # Rebuild full output along feature dimension
    return torch.cat(shards, dim=-1)

We also need to define a way to initalize the sharded parameters.

### Column Parallel Initialization:

Full weight shape:
- (out_features, in_features)
Local shard shape:
- (out_features // tp_size, in_features)


### Row Parallel Initialization

Full weight shape:
- (out_features, in_features)
Local shard shape:
- (out_features, in_features // tp_size)

In [18]:
# column wise

def reset_parameters(self):
        
    master_weight = torch.empty(self.out_features, self.in_features)

    nn.init.uniform_(master_weight, -0.1, 0.1)
    
    shards = master_weight.chunk(self.tp_size, dim=0)
    self.weight.data.copy_(shards[self.tp_rank])


# row wise

def reset_parameters(self):

    master_weight = torch.empty(self.out_features, self.in_features)

    nn.init.uniform_(master_weight, -0.1, 0.1)
    
    shards = master_weight.chunk(self.tp_size, dim=1)
    self.weight.data.copy_(shards[self.tp_rank])

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class ColumnParallelLinear(nn.Module):

    def __init__(self, in_features, out_features, tp_size, tp_rank, gather_output=False):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.tp_size = tp_size
        self.tp_rank = tp_rank
        self.gather_output = gather_output

        # Each rank owns only part of the output rows.
        assert out_features % tp_size == 0
        self.out_per_rank = out_features // tp_size

        # Local shard: (out_per_rank, in_features)
        self.weight = nn.Parameter(torch.empty(self.out_per_rank, in_features))

        # Initialize the local shard here
        self.reset_parameters()


    def gather_output_shards(self, y_local):
    
        # Make one empty slot per rank, each same shape as local shard
        shards = [torch.empty_like(y_local) for _ in range(self.tp_size)]

        # Collect y_local from every rank into shards
        dist.all_gather(shards, y_local)

        # Rebuild full output along feature dimension
        return torch.cat(shards, dim=-1)

    
    def forward(self, x):

        # Local output shard
        y_local = F.linear(x, self.weight)
        
        # Keep sharded output unless full output is requested
        if not self.gather_output:
            return y_local
        return self.gather_output_shards(y_local)
    

    def reset_parameters(self):
        
        master_weight = torch.empty(self.out_features, self.in_features)

        nn.init.uniform_(master_weight, -0.1, 0.1)
        
        shards = master_weight.chunk(self.tp_size, dim=0)
        self.weight.data.copy_(shards[self.tp_rank])

Similarly, we can write out the idea for row parallel now.

In [20]:
class RowParallelLinear(nn.Module):

    def __init__(self, in_features, out_features, tp_size, tp_rank):

        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.tp_size = tp_size
        self.tp_rank = tp_rank
        
        # Each rank owns only part of the input columns.
        assert in_features % tp_size == 0
        self.in_per_rank = in_features // tp_size

        # Local shard: (out_features, in_per_rank)
        self.weight = nn.Parameter(torch.empty(out_features, self.in_per_rank))

        # Initialize the local shard here
        self.reset_parameters()


    def reduce_output(self, y_partial):
        
        # Sum partial outputs across all ranks.
        dist.all_reduce(y_partial)
        return y_partial
    

    def forward(self, x):

        # x is already sharded: (B, S, in_per_rank)
        y_partial = F.linear(x, self.weight)

        # Each rank computed a partial full-width output.
        # Sum them to get the true output.
        return self.reduce_output(y_partial)
    

    def reset_parameters(self):

        master_weight = torch.empty(self.out_features, self.in_features)

        nn.init.uniform_(master_weight, -0.1, 0.1)
        
        shards = master_weight.chunk(self.tp_size, dim=1)
        self.weight.data.copy_(shards[self.tp_rank])

## Tensor Parallism On Mini GPT

Our original implementation couldn't be used for tensor parallel (like using pytorch's native multi-head attention), so we've remade a tensor parallel comptable version.

We changed:

- the attention implementation from PyTorch's packaged nn.MultiheadAttention to an explicit Attention module with separate q_proj, k_proj, v_proj, and out_proj linear layers
- the feedforward block into explicit projection layers so tensor parallel can replace them cleanly
- the model structure to expose Picotron-style insertion points like decoder_layers, attention, mlp, and final_proj
- the linear layers used at those insertion points with tensor-parallel-compatible ColumnParallelLinear and RowParallelLinear modules
- the final projection so logits can be gathered back across ranks when needed

In [27]:
from mini_gpt_tp import ModelConfig, MiniTransformerTPFriendly

model = MiniTransformerTPFriendly(ModelConfig())

And now, since we already defined the TP-capable building blocks, ColumnParallelLinear and RowParallelLinear, we are making 2 helper functions that take one dense layer and replace it with the corresponding TP version.

The model still starts life as a normal dense model with ordinary nn.Linear layers.

In [ ]:
def _shard_column_linear(
    linear: nn.Linear, tp_size: int, tp_rank: int, gather_output: bool = False
):
    new_linear = ColumnParallelLinear(
        in_features=linear.in_features,
        out_features=linear.out_features,
        tp_size=tp_size,
        tp_rank=tp_rank,
        gather_output=gather_output,
    )
    new_linear.weight.data.copy_(linear.weight.data.chunk(tp_size, dim=0)[tp_rank])
    return new_linear


def _shard_row_linear(linear: nn.Linear, tp_size: int, tp_rank: int):
    new_linear = RowParallelLinear(
        in_features=linear.in_features,
        out_features=linear.out_features,
        tp_size=tp_size,
        tp_rank=tp_rank,
    )
    new_linear.weight.data.copy_(linear.weight.data.chunk(tp_size, dim=1)[tp_rank])
    return new_linear

This is just the full applying tensor parallel to the mini_gpt model

In [28]:
def apply_tensor_parallel(model: MiniTransformerTPFriendly, tp_size: int, tp_rank: int):
    assert model.final_proj.out_features % tp_size == 0

    for layer in model.decoder_layers:
        assert layer.attention.num_heads % tp_size == 0
        assert layer.mlp.up_proj.out_features % tp_size == 0

        # q/k/v keep their outputs sharded across ranks.
        layer.attention.q_proj = _shard_column_linear(
            layer.attention.q_proj, tp_size, tp_rank
        )
        layer.attention.k_proj = _shard_column_linear(
            layer.attention.k_proj, tp_size, tp_rank
        )
        layer.attention.v_proj = _shard_column_linear(
            layer.attention.v_proj, tp_size, tp_rank
        )

        # out_proj consumes sharded heads and reduces partial outputs.
        layer.attention.out_proj = _shard_row_linear(
            layer.attention.out_proj, tp_size, tp_rank
        )

        # MLP uses the same column-then-row pattern.
        layer.mlp.up_proj = _shard_column_linear(layer.mlp.up_proj, tp_size, tp_rank)
        layer.mlp.down_proj = _shard_row_linear(layer.mlp.down_proj, tp_size, tp_rank)

    # Final logits need to be gathered back to full vocab.
    model.final_proj = _shard_column_linear(
        model.final_proj, tp_size, tp_rank, gather_output=True
    )
    return model

As we are currently on a CPU... well, we just have tp_size of 1 and rank of 0.

In [29]:
model = apply_tensor_parallel(model, tp_size=1, tp_rank=0)
print(model)

MiniTransformerTPFriendly(
  (embedding): Embedding(1000, 256)
  (decoder_layers): ModuleList(
    (0-3): 4 x DecoderLayer(
      (attention): Attention(
        (q_proj): ColumnParallelLinear()
        (k_proj): ColumnParallelLinear()
        (v_proj): ColumnParallelLinear()
        (out_proj): RowParallelLinear()
      )
      (mlp): MLP(
        (up_proj): ColumnParallelLinear()
        (down_proj): RowParallelLinear()
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (final_proj): ColumnParallelLinear()
)
